# Results Visualization: Model Performance Analysis

This notebook visualizes training results and model performance metrics.

**Key Objectives:**
- Display training curves (loss, accuracy)
- Visualize confusion matrix
- Analyze per-class metrics
- Generate comprehensive performance report

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")

## 1. Load Training Metrics

In [ ]:
results_dir = Path('results')
metrics_file = results_dir / 'training_metrics.json'

if metrics_file.exists():
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    # Convert to DataFrame
    metrics_df = pd.DataFrame(metrics)
    
    print(f"Loaded metrics for {len(metrics_df)} epochs")
    print(f"\nColumns: {metrics_df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(metrics_df.head())
else:
    print(f"Metrics file not found: {metrics_file}")
    print("Run training first: python training/train.py")

## 2. Training Curves

In [ ]:
if 'metrics_df' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Training loss
    ax = axes[0, 0]
    ax.plot(metrics_df['epoch'], metrics_df['train_loss'], marker='o', linewidth=2, label='Training', color='#3498db')
    ax.plot(metrics_df['epoch'], metrics_df['val_loss'], marker='s', linewidth=2, label='Validation', color='#e74c3c')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Training accuracy
    ax = axes[0, 1]
    ax.plot(metrics_df['epoch'], metrics_df['train_acc']*100, marker='o', linewidth=2, label='Training', color='#3498db')
    ax.plot(metrics_df['epoch'], metrics_df['val_acc']*100, marker='s', linewidth=2, label='Validation', color='#e74c3c')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Training and Validation Accuracy', fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_ylim([0, 105])
    ax.grid(True, alpha=0.3)
    
    # Learning rate
    ax = axes[1, 0]
    ax.plot(metrics_df['epoch'], metrics_df['learning_rate'], marker='o', linewidth=2, color='#2ecc71')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    
    # Gradient norm (if available)
    if 'grad_norm' in metrics_df.columns:
        ax = axes[1, 1]
        ax.plot(metrics_df['epoch'], metrics_df['grad_norm'], marker='o', linewidth=2, color='#f39c12')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Gradient Norm')
        ax.set_title('Gradient Norm per Epoch', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/training_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Training curves saved to results/training_curves.png")

## 3. Load Test Metrics

## 3. Load Test Metrics

In [ ]:
test_metrics_file = results_dir / 'test_metrics.json'
per_class_file = results_dir / 'per_class_metrics.csv'

if test_metrics_file.exists():
    with open(test_metrics_file, 'r') as f:
        test_metrics = json.load(f)
    
    print("\nTest Metrics:")
    print("="*60)
    for key, value in test_metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value*100:.2f}%" if key.endswith('_') or 'acc' in key else f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
else:
    print(f"Test metrics file not found: {test_metrics_file}")

if per_class_file.exists():
    per_class_df = pd.read_csv(per_class_file)
    print(f"\nPer-class metrics loaded for {len(per_class_df)} classes")
    print("\nTop 5 classes by F1-score:")
    print(per_class_df.nlargest(5, 'f1_score')[['class_name', 'precision', 'recall', 'f1_score']])
else:
    print(f"Per-class metrics file not found: {per_class_file}")

## 4. Per-Class Performance Analysis

In [ ]:
if 'per_class_df' in locals():
    # Sort by F1-score
    per_class_sorted = per_class_df.sort_values('f1_score')
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # F1-scores
    ax = axes[0, 0]
    ax.barh(range(len(per_class_sorted)), per_class_sorted['f1_score'], color='#3498db')
    ax.set_yticks(range(len(per_class_sorted)))
    ax.set_yticklabels(per_class_sorted['class_name'], fontsize=8)
    ax.set_xlabel('F1-Score')
    ax.set_title('Per-Class F1-Scores (sorted)', fontsize=12, fontweight='bold')
    ax.set_xlim([0, 1.05])
    ax.grid(True, alpha=0.3, axis='x')
    
    # Precision vs Recall
    ax = axes[0, 1]
    scatter = ax.scatter(per_class_df['recall'], per_class_df['precision'], 
                         s=per_class_df['support']*10, alpha=0.6, c=per_class_df['f1_score'], 
                         cmap='viridis', edgecolors='black', linewidth=0.5)
    ax.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Perfect')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision vs Recall (bubble size = support)', fontsize=12, fontweight='bold')
    ax.set_xlim([0, 1.05])
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('F1-Score')
    
    # Support distribution
    ax = axes[1, 0]
    support_sorted = per_class_df.sort_values('support', ascending=False).head(15)
    ax.bar(range(len(support_sorted)), support_sorted['support'], color='#e74c3c')
    ax.set_xticks(range(len(support_sorted)))
    ax.set_xticklabels(support_sorted['class_name'], rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Number of Samples')
    ax.set_title('Top 15 Classes by Sample Count', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Metrics distribution
    ax = axes[1, 1]
    ax.hist(per_class_df['precision'], bins=15, alpha=0.5, label='Precision', color='#3498db', edgecolor='black')
    ax.hist(per_class_df['recall'], bins=15, alpha=0.5, label='Recall', color='#e74c3c', edgecolor='black')
    ax.hist(per_class_df['f1_score'], bins=15, alpha=0.5, label='F1-Score', color='#2ecc71', edgecolor='black')
    ax.set_xlabel('Score')
    ax.set_ylabel('Number of Classes')
    ax.set_title('Distribution of Per-Class Metrics', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('results/per_class_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Per-class analysis plots saved to results/per_class_analysis.png")

## 5. Confusion Matrix Visualization

In [ ]:
confusion_matrix_file = results_dir / 'confusion_matrix.png'

if confusion_matrix_file.exists():
    from PIL import Image
    
    img = Image.open(confusion_matrix_file)
    plt.figure(figsize=(14, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print("✓ Confusion matrix loaded and displayed")
else:
    print(f"Confusion matrix file not found: {confusion_matrix_file}")

## 6. Classification Report

## 6. Classification Report

In [ ]:
report_file = results_dir / 'classification_report.txt'

if report_file.exists():
    with open(report_file, 'r') as f:
        report_text = f.read()
    
    print("\nClassification Report:")
    print("="*80)
    print(report_text)
    print("="*80)
else:
    print(f"Classification report file not found: {report_file}")

## 7. Performance Summary

In [ ]:
if 'test_metrics' in locals() and 'metrics_df' in locals():
    print("\n" + "="*80)
    print("PERFORMANCE SUMMARY")
    print("="*80)
    
    summary = f"""
TRAINING RESULTS:
  Total epochs trained: {len(metrics_df)}
  Best validation loss: {metrics_df['val_loss'].min():.4f} (epoch {metrics_df['val_loss'].idxmin() + 1})
  Best validation accuracy: {metrics_df['val_acc'].max()*100:.2f}% (epoch {metrics_df['val_acc'].idxmax() + 1})
  Final training loss: {metrics_df['train_loss'].iloc[-1]:.4f}
  Final validation loss: {metrics_df['val_loss'].iloc[-1]:.4f}

TEST RESULTS:
  Overall accuracy: {test_metrics.get('accuracy', 0)*100:.2f}%
  Macro precision: {test_metrics.get('macro_precision', 0)*100:.2f}%
  Macro recall: {test_metrics.get('macro_recall', 0)*100:.2f}%
  Macro F1: {test_metrics.get('macro_f1', 0)*100:.2f}%

PER-CLASS STATISTICS:
  Mean F1-score: {per_class_df['f1_score'].mean()*100:.2f}%
  Median F1-score: {per_class_df['f1_score'].median()*100:.2f}%
  Std deviation: {per_class_df['f1_score'].std()*100:.2f}%
  Min F1-score: {per_class_df['f1_score'].min()*100:.2f}%
  Max F1-score: {per_class_df['f1_score'].max()*100:.2f}%

WORST PERFORMING CLASSES:
"""
    
    print(summary)
    
    worst_classes = per_class_df.nsmallest(5, 'f1_score')[['class_name', 'f1_score', 'precision', 'recall']]
    for idx, row in worst_classes.iterrows():
        print(f"  {row['class_name']}: F1={row['f1_score']:.2f} | Precision={row['precision']:.2f} | Recall={row['recall']:.2f}")
    
    print("\n" + "="*80)
else:
    print("Unable to generate summary - required data not loaded")

---

**Notebook Complete** ✓

This notebook provides comprehensive visualization of model training and testing results including:
- Training curves (loss, accuracy, learning rate)
- Per-class performance analysis
- Confusion matrix visualization
- Classification report
- Performance summary

All visualizations have been saved to the `results/` directory.